# Partie 3 - Referentiel stations, jointures, `explain(True)` (Personne 2)

In [ ]:
import os, sys, time
from pathlib import Path
sys.path.append(os.environ.get("PEDALI_DATA_MODULE", "../data"))
from pedali_schema import *
from pyspark.sql import SparkSession, functions as F

# Jupyter (docker) : SPARK_MASTER=spark://spark-master:7077 ; en local : local[*]
spark = (SparkSession.builder.appName("pedali-p2")
         .master(os.environ.get("SPARK_MASTER", "local[*]"))
         .config("spark.sql.shuffle.partitions", "8")   # donnees petites : 200 partitions (defaut) = surcout inutile
         .getOrCreate())
DATA = Path(os.environ.get("PEDALI_DATA", "../data")).resolve()
print(spark.version, spark.sparkContext.master, spark.sparkContext.defaultParallelism)

In [ ]:
trips = read_trips(spark, str(DATA / "trips_history.jsonl"))
stations = read_stations(spark, str(DATA / "stations.json"))
clean = clean_trips(trips, stations).filter("status = 'completed'")
stations.show(truncate=False)

## 1. Jointure trajets / referentiel stations
On enrichit chaque trajet avec le nom de la station de depart et la ville du referentiel.

In [ ]:
s = stations.select(F.col("station_id").alias("start_station_id"), F.col("name").alias("start_name"), F.col("capacity"))
enriched = clean.join(s, "start_station_id")
enriched.select("trip_id", "start_station_id", "start_name", "city", "duration_min", "price_mad").show(5)
print("lignes avant/apres jointure :", clean.count(), enriched.count())

## 2. Jointure + agregation par station, et plan Catalyst

In [ ]:
by_station = (enriched.groupBy("start_station_id", "start_name", "city")
    .agg(F.count("*").alias("trajets"), F.round(F.sum("price_mad"), 2).alias("revenu"),
         F.round(F.avg("duration_min"), 1).alias("duree_moy"))
    .orderBy(F.desc("trajets")))
by_station.show(12, truncate=False)

In [ ]:
by_station.explain(True)

### Lecture du plan
- **Parsed / Analyzed Logical Plan** : l'arbre tel qu'ecrit, puis avec colonnes/types resolus.
- **Optimized Logical Plan** : Catalyst pousse les filtres (`status = completed`, `user_id IS NOT NULL`, `duration_min >= 0`) au plus pres de la source et ne lit que les colonnes utiles (*column pruning*).
- **Physical Plan** : la table `stations` etant minuscule, Spark choisit un **BroadcastHashJoin** (le referentiel est diffuse aux executors, pas de shuffle du gros cote) ; les agregations sont faites en deux temps (`HashAggregate` partiel, `Exchange`, `HashAggregate` final).

In [ ]:
from pyspark.sql.functions import broadcast
# Forcer / desactiver le broadcast pour comparer les plans physiques
print("--- sans broadcast (seuil a -1) ---")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
clean.join(s, "start_station_id").groupBy("start_name").count().explain()
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10 * 1024 * 1024)
print("--- avec broadcast ---")
clean.join(broadcast(s), "start_station_id").groupBy("start_name").count().explain()

## 3. Comparaison RDD / DataFrame (meme calcul : trajets par ville)

In [ ]:
def timed(f, n=3):
    best = 1e9
    for _ in range(n):
        t = time.perf_counter(); r = f(); best = min(best, time.perf_counter() - t)
    return best, r

clean.cache().count()
t_rdd, r_rdd = timed(lambda: sorted(clean.rdd.map(lambda r: (r.city, 1)).reduceByKey(lambda a, b: a + b).collect()))
t_df, r_df = timed(lambda: sorted((r.city, r["count"]) for r in clean.groupBy("city").count().collect()))
assert r_rdd == r_df
print(f"RDD       : {t_rdd*1000:7.1f} ms  {r_rdd}")
print(f"DataFrame : {t_df*1000:7.1f} ms  {r_df}")

**Conclusion** : meme resultat ; le DataFrame beneficie de l'optimiseur Catalyst et de Tungsten (execution sur `UnsafeRow`, sans serialisation Python par ligne), alors que le RDD execute des lambdas Python ligne par ligne. Attention : l'ecart mesure ici est amplifie par l'execution locale sous Windows (chaque tache RDD lance un process Python) ; sur le cluster Docker (Linux) il sera plus faible, mais le sens reste le meme. A re-mesurer sur le cluster pour le rapport.